# Paper-ready CV summary table

Load one completed CV experiment and export a booktabs LaTeX table for the 15-bp realised-depeg definition and the chosen false-alert budget. Operational metrics include event/calendar-block bootstrap 95% confidence intervals; AUC and AUPRC are secondary row-level ranking metrics.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'cv_model_comparison.py').exists():
    PROJECT_DIR = PROJECT_DIR / '8. Early-Warning Model'
if not (PROJECT_DIR / 'cv_model_comparison.py').exists():
    raise FileNotFoundError('Run this notebook from the repository root or 8. Early-Warning Model.')

# ---- Edit these values ----------------------------------------------------
EXPERIMENT_NAME = 'cv_model_comparison_YYYY-MM-DD_15bp'
DEPEG_THRESHOLD_BPS = 15.0
FALSE_ALERT_BUDGET = 2.0
LOG_DIR = PROJECT_DIR / 'lightning_logs'
EXPERIMENT_DIR = LOG_DIR / EXPERIMENT_NAME
if not EXPERIMENT_DIR.exists():
    available = sorted(p.name for p in LOG_DIR.glob('cv_model_comparison*') if p.is_dir())
    raise FileNotFoundError(f'{EXPERIMENT_DIR} does not exist. Available experiments: {available[-10:]}')
OUTPUT_DIR = EXPERIMENT_DIR / 'paper_ready'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def read_latest(files, keys):
    frames = []
    for path in files:
        frame = pd.read_csv(path)
        if not frame.empty:
            frame['_source_mtime'] = path.stat().st_mtime
            frames.append(frame)
    if not frames:
        raise FileNotFoundError(f'No matching CV reports found under {EXPERIMENT_DIR}')
    return (pd.concat(frames, ignore_index=True).sort_values('_source_mtime')
            .drop_duplicates(keys, keep='last').drop(columns='_source_mtime'))

primary = read_latest(
    EXPERIMENT_DIR.glob('*_experiment_summary*/artifacts/comparison/model_comparison_summary.csv'),
    ['target_threshold', 'alpha', 'model_name'])
budget = read_latest(
    EXPERIMENT_DIR.glob('*/artifacts/cv/utility_by_false_alert_budget_summary.csv'),
    ['target_threshold', 'alpha', 'model_name', 'false_alert_budget_per_month'])

primary = primary[np.isclose(pd.to_numeric(primary.target_threshold), DEPEG_THRESHOLD_BPS)].copy()
budget = budget[
    np.isclose(pd.to_numeric(budget.target_threshold), DEPEG_THRESHOLD_BPS)
    & np.isclose(pd.to_numeric(budget.false_alert_budget_per_month), FALSE_ALERT_BUDGET)
].copy()
if primary.empty or budget.empty:
    raise ValueError('No rows match the requested depeg threshold and false-alert budget.')

auc_cols = ['target_threshold', 'alpha', 'model_name', 'cv_auc_mean', 'cv_auc_std', 'cv_auprc_mean', 'cv_auprc_std']
results = budget.merge(primary[auc_cols], on=['target_threshold', 'alpha', 'model_name'], how='left')
results = results.sort_values(['cv_event_utility_score_mean', 'cv_timely_event_recall_mean'], ascending=False).reset_index(drop=True)
display(results)


In [ ]:
MODEL_LABELS = {'xgboost': 'XGBoost', 'lightgbm': 'LightGBM', 'catboost': 'CatBoost', 'random_forest': 'Random forest'}

def estimate_ci(row, metric, digits=3):
    point = row.get(f'cv_{metric}_mean', np.nan)
    low, high = row.get(f'cv_{metric}_ci_lower', np.nan), row.get(f'cv_{metric}_ci_upper', np.nan)
    if pd.isna(point):
        return 'NA'
    if pd.notna(low) and pd.notna(high):
        return f'{point:.{digits}f} [{low:.{digits}f}, {high:.{digits}f}]'
    return f'{point:.{digits}f}'

table = pd.DataFrame({
    'Model': results.model_name.map(MODEL_LABELS).fillna(results.model_name),
    r'$\alpha$': results.alpha.map(lambda x: f'{x:g}'),
    'Event utility': results.apply(estimate_ci, axis=1, metric='event_utility_score'),
    'Timely event recall': results.apply(estimate_ci, axis=1, metric='timely_event_recall'),
    'False alerts/month': results.apply(lambda r: estimate_ci(r, 'false_alerts_per_month', 2), axis=1),
    'Median lead (h)': results.apply(lambda r: estimate_ci(r, 'median_lead_hours', 1), axis=1),
    'AUC': results.cv_auc_mean.map(lambda x: f'{x:.3f}' if pd.notna(x) else 'NA'),
    'AUPRC': results.cv_auprc_mean.map(lambda x: f'{x:.3f}' if pd.notna(x) else 'NA'),
})
display(table)


In [ ]:
caption = (
    f'Operational out-of-sample comparison at the {DEPEG_THRESHOLD_BPS:g}-basis-point depeg '
    f'threshold and a budget of at most {FALSE_ALERT_BUDGET:g} false-alert episodes per month. '
    'Probability thresholds are selected on the preceding validation fold. Event utility is the '
    'primary selection metric; brackets report event/calendar-block bootstrap 95\\% confidence intervals. '
    'AUC and AUPRC are secondary row-level ranking metrics.'
)
latex = table.to_latex(
    index=False, escape=False, column_format='llrrrrrr', position='!htbp',
    caption=caption, label='tab:cv_model_alpha_results')
latex = latex.replace('\\centering\n', '\\centering\n\\small\n', 1)

tex_path = OUTPUT_DIR / 'table_cv_model_alpha_results.tex'
csv_path = OUTPUT_DIR / 'table_cv_model_alpha_results.csv'
tex_path.write_text(latex)
table.to_csv(csv_path, index=False)
display(Markdown(f'Wrote `{tex_path}` and `{csv_path}`.'))
print(latex)
